# Classical Solver for TSP and its Variants using Google's CP-SAT.

In [2]:
from ortools.sat.python import cp_model
import numpy as np

In [ ]:
def solve_tsp_cpsat(distance_matrix):
    n = len(distance_matrix)
    model  = cp_model.CpModel()
    solver = cp_model.CpSolver()

    # Binary arc variables x[i][j] = 1 if arc i→j is in the tour
    x = {}
    arcs = []
    for i in range(n):
        for j in range(n):
            if i != j:
                b = model.NewBoolVar(f"x_{i}_{j}")
                x[(i, j)] = b
                arcs.append((i, j, b))

    # Circuit constraint 
    model.AddCircuit(arcs)

    # Obj: minimize total distance 
    model.Minimize(
        sum(distance_matrix[i][j] * x[(i, j)]
            for i in range(n)
            for j in range(n) if i != j)
    )

    # Solve 
    solver.parameters.log_search_progress = True  
    status = solver.Solve(model)

    # Extract solution 
    if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        # Reconstruct tour 
        next_node = {}
        for (i, j), var in x.items():
            if solver.Value(var) == 1:
                next_node[i] = j

        tour = [0]
        while next_node[tour[-1]] != 0:
            tour.append(next_node[tour[-1]])
        tour.append(0)  # return to depot

        print(f"Status : {solver.StatusName(status)}")
        print(f"Length : {solver.ObjectiveValue()}")
        print(f"Tour   : {tour}")
        return tour

    print("No solution found.")
    return None

In [4]:
test = [
    [0, 5, 1, 3],
    [5, 0, 4, 2],
    [1, 4, 0, 2],
    [3, 2, 2, 0]
]

solve_tsp_cpsat(test)


Starting CP-SAT solver v9.15.6755
Parameters: log_search_progress: true
Setting number of workers to 8

Initial optimization model '': (model_fingerprint: 0x505e1b74db2de9da)
#Variables: 12 (#bools: 12 in objective) (12 primary variables)
  - 12 Booleans in [0,1]
#kCircuit: 1

Starting presolve at 0.00s
  6.50e-05s  0.00e+00d  [DetectDominanceRelations] 
  1.47e-03s  2.00e-08d  [PresolveToFixPoint] #num_loops=2 #num_dual_strengthening=1 
  3.70e-05s  0.00e+00d  [ExtractEncodingFromLinear] 
  3.00e-06s  0.00e+00d  [DetectDuplicateColumns] 
  8.90e-05s  0.00e+00d  [DetectDuplicateConstraints] 
[Symmetry] Graph for symmetry has 28 nodes and 36 arcs.
[Symmetry] Symmetry computation done. time: 3.7e-05 dtime: 3.16e-06
  5.00e-06s  0.00e+00d  [DetectDuplicateConstraintsWithDifferentEnforcements] 
  1.96e-04s  1.38e-05d  [Probe] #probed=24 #new_binary_clauses=6 
  0.00e+00s  0.00e+00d  [MaxClique] 
  9.00e-06s  0.00e+00d  [DetectDominanceRelations] 
  1.06e-04s  1.00e-08d  [PresolveToFixPoin

[0, 2, 1, 3, 0]